# CS 687 · Homework 5 · Project 2, final checkpoint

**DDIM, flow matching, and sample quality against network calls**

> **Before you begin:** Read the repository's `README.md`. It contains the grading and submission rules, including which parts of this notebook you may edit, how to keep the structure that automatic grading depends on, and the required file name of your submission. Read it before changing any cell.

This notebook follows three parts of the Lecture 4 notes, "The cost of a thousand steps", "A deterministic sampler" and "Determinism and its costs", and the Lecture 5 notes from "The path of the noising process" to "Straight paths and few steps". It is meant to run in Google Colab. You may instead run it on your own computer if you can set up the required Python and Jupyter environment yourself. Everything on the two clusters runs on the processor in a minute or two. Section 8 trains a network on handwritten digits and needs a GPU, so choose a GPU runtime under Runtime > Change runtime type before you start.

The homework contains five programming tasks, four short guided calculations and four report responses. Every code answer cell is marked clearly and followed by an immediate check, so you can correct it before continuing. The four report cells are grouped near the end. The final check reruns the guided calculations and the complete public test suite; always run it before you submit.

Project 2 ends with one measurement: how the quality of samples of the two clusters falls as the number of network calls per sample falls from a thousand to one. Homework 4 built the noise predictor and the ancestral sampler, which needs a thousand calls. This checkpoint adds DDIM, which samples the same network in a few calls, and flow matching, which trains a network on straight lines and samples it with Euler steps. Then it measures all three samplers on networks you train.

## 1 · Setup

If you are using Colab, the cell below clones the homework repository from GitHub, which holds the code, the data, the text and the tests the notebook uses.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_NAME = "cs687-hw05-p2-final"
REPO_URL = "https://github.com/LiRA-Lab/cs687-hw05-p2-final.git"
RELEASE = "v1.0"

try:
    import google.colab  # type: ignore
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.run(
            ["git", "-c", "advice.detachedHead=false", "clone", "--branch", RELEASE,
             "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
else:
    repo_root = Path.cwd().resolve()
    if repo_root.name == "notebooks":
        repo_root = repo_root.parent
    if not (repo_root / "notebook_support.py").is_file():
        raise RuntimeError(
            "Could not find notebook_support.py. Open this notebook from "
            "somewhere inside the Homework 5 repository."
        )

sys.path.insert(0, str(repo_root))
from notebook_support import prepare_notebook
repo_root = prepare_notebook(repo_root)

## 2 · Where Homework 4 stopped

Homework 4 noised the two clusters, trained a small network to predict the added noise, and generated samples with ancestral sampling, one call of the network for each of the thousand levels. This repository ships the course's code of that homework complete: the schedule in `cs687/schedule.py`, the networks in `cs687/networks.py`, the two clusters with their exact noise prediction and the three statistics in `cs687/toy.py`, and the noisy image at any level and the loss of DDPM in `cs687/diffusion.py`.

The cell retrains the small noise predictor of Homework 4 with the same settings, 4,000 steps of Adam at the learning rate 0.001 on batches of 512, in under half a minute on the processor. Then it samples 2,000 points with a thousand calls, with the exact noise prediction and with that network, from the same 2,000 draws of the noise, which every sampler of this notebook starts from. The three statistics are Homework 4's: the share of samples with a positive first coordinate, 0.5 for the data; the average distance of the first coordinate from zero, the middle line, about 2 for the data; and the spread of the first coordinate around the mean of its own half, about 0.5 for the data.

In [ ]:
import math
import time

import torch

from cs687 import (
    ExactNoisePredictor, Schedule, ToyNoisePredictor, cluster_statistics, even_levels,
    sample_ancestral, sample_clusters, train_run,
)
from notebook_support import pick_device, plot_points

device = pick_device()
print("device for the digits:", device, "   the two clusters always run on the processor")

schedule = Schedule()
exact_noise = ExactNoisePredictor(schedule)

torch.manual_seed(0)
noise_network = ToyNoisePredictor()
started = time.time()
noise_curve = train_run(noise_network, sample_clusters, "noise", steps=4000, batch_size=512, learning_rate=1e-3,
                        generator=torch.Generator().manual_seed(3), schedule=schedule, print_every=4000)
print(f"the noise predictor trained in {time.time() - started:.1f} s")

def show(name, stats, calls=None):
    cost = f"   {calls:4d} call{'s' if calls != 1 else ' '}" if calls is not None else ""
    print(f"{name:38s} share right {stats['share_right']:.3f}   mean distance {stats['mean_distance']:.3f}   "
          f"spread within {stats['spread_within']:.3f}{cost}")

data_stats = cluster_statistics(sample_clusters(2000, torch.Generator().manual_seed(5)))
x_start = torch.randn(2000, 2, generator=torch.Generator().manual_seed(6))
show("the data", data_stats)
ancestral_exact, calls = sample_ancestral(exact_noise, schedule, x_start, even_levels(1000), torch.Generator().manual_seed(7))
show("ancestral, the exact prediction", cluster_statistics(ancestral_exact), calls)
ancestral_network, calls = sample_ancestral(noise_network, schedule, x_start, even_levels(1000), torch.Generator().manual_seed(7))
show("ancestral, your noise predictor", cluster_statistics(ancestral_network), calls)
plot_points([("the starting noise", x_start), ("1000 calls, exact prediction", ancestral_exact),
             ("1000 calls, your noise predictor", ancestral_network)])

## 3 · DDIM

The Lecture 4 notes' part "The cost of a thousand steps" tries the obvious way to spend fewer calls: jump from level $t$ to a level $s$ far below it, the level that the notes write $t-\Delta t$, drawing the lower level from the Gaussian step back given the clean image, with the estimate of the clean image in its place. The supplied `ancestral_jump` in `cs687/diffusion.py` is that step, and `sample_ancestral` in `cs687/samplers.py` runs it down a list of levels; with all thousand levels it is the sampler of Homework 4.

The subsection "Large steps from the estimate of the clean image" builds a deterministic step instead. At the current level the network predicts the noise $\hat{\boldsymbol\epsilon}=\hat{\boldsymbol\epsilon}_{\boldsymbol\theta}(\mathbf x_t,t)$, which gives the estimate of the clean image
$$\hat{\mathbf x}_0=\frac{\mathbf x_t-\sqrt{1-\bar\alpha_t}\,\hat{\boldsymbol\epsilon}}{\sqrt{\bar\alpha_t}},$$
and the step combines this estimate and the same predicted noise with the numbers of the lower level,
$$\mathbf x_s=\sqrt{\bar\alpha_s}\,\hat{\mathbf x}_0+\sqrt{1-\bar\alpha_s}\,\hat{\boldsymbol\epsilon}.$$
No fresh noise is drawn. This is deterministic DDIM. Divided by the weight of the clean image, the point becomes the rescaled image $\bar{\mathbf x}_t=\mathbf x_t/\sqrt{\bar\alpha_t}$ with the noise scale $\sigma_t=\sqrt{(1-\bar\alpha_t)/\bar\alpha_t}$, and the same step reads $\bar{\mathbf x}_s=\bar{\mathbf x}_t+(\sigma_s-\sigma_t)\,\hat{\boldsymbol\epsilon}$, an Euler step in the noise scale.

### One DDIM step by hand · 4 points

Take one coordinate at level 600 with the value $x_{600}=0.6$, and suppose the network predicts the noise $\hat\epsilon=0.3$ there. Compute the step to level 300 two ways, with the numbers of `exact_schedule`, the copy of the schedule in double precision that the first line of the cell makes:

1. `x0_hat_600`: the estimate of the clean value at level 600.
2. `x_300`: the value at level 300, from the estimate and the predicted noise.
3. `sigma_600` and `sigma_300`: the noise scales at the two levels.
4. `x_300_rescaled`: the same value at level 300 from the rescaled form: rescale $x_{600}$, take the Euler step in the noise scale, and multiply back by the weight of the clean image at level 300.

**Write the five lines in the cell below.**

In [ ]:
exact_schedule = schedule.double()
# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Compute x0_hat_600, x_300, sigma_600, sigma_300 and x_300_rescaled.")
# ================= YOUR CODE ENDS HERE ===================
print(f"x0_hat_600 {x0_hat_600:.4f}   x_300 {x_300:.4f}")
print(f"sigma_600 {sigma_600:.4f}   sigma_300 {sigma_300:.4f}   x_300_rescaled {x_300_rescaled:.4f}")

In [ ]:
from notebook_support import check_ddim_by_hand

check_ddim_by_hand(schedule, x0_hat_600, x_300, sigma_600, sigma_300, x_300_rescaled)

### Task 1 · one DDIM step · 7 points

Write `ddim_step(model, x_t, t, s, schedule)`, step 2 of the notes' algorithm "sample with deterministic DDIM". It receives a noise predictor called as `model(x_t, levels)`, a batch `x_t` whose rows are all at level `t`, the current level `t` and a lower level `s` as Python integers with $0\le s<t\le T$, and the schedule, and it returns the batch at level `s`, of the shape of `x_t`. The two levels need not be neighbours. Two details are fixed, and the tests check them:

- Call the network once, as `model(x_t, levels)`, with `levels` a long tensor of $B$ copies of `t` on the device of `x_t`.
- Draw no noise, and leave the tensor `x_t` unchanged.

The schedule holds $\bar\alpha_0=1$ at index 0, as in Homework 4. The supplied loop `sample_ddim` in `cs687/samplers.py` calls your step down a list of levels under `torch.no_grad()` and counts the calls. The docstring of the stub in `cs687/ddim.py` states the contract.

In [ ]:
import torch


def ddim_step(model, x_t, t, s, schedule):
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 1: implement ddim_step")
    # ================= YOUR CODE ENDS HERE ===================
print("function defined")

In [ ]:
from notebook_support import run_task1_tests

run_task1_tests(ddim_step)

### DDIM against the larger ancestral steps

The cell samples the two clusters from the same 2,000 draws with the larger ancestral steps and with DDIM, at 1000, 50, 20, 10 and 5 calls on evenly spaced levels, first with the exact noise prediction, so that what you see is the sampler and nothing else, and then with your noise predictor. `even_levels(calls)` gives the evenly spaced levels from 1000 down to 0. Keep the output: the report reads it.

In [ ]:
from cs687 import sample_ddim

panels = []
for source, model in (("the exact prediction", exact_noise), ("your noise predictor", noise_network)):
    print(f"--- with {source}")
    for calls in (1000, 50, 20, 10, 5):
        ancestral, used = sample_ancestral(model, schedule, x_start, even_levels(calls), torch.Generator().manual_seed(7))
        show("ancestral steps", cluster_statistics(ancestral), used)
        ddim_samples, used = sample_ddim(model, schedule, x_start, even_levels(calls))
        show("DDIM", cluster_statistics(ddim_samples), used)
        if calls == 10 and model is exact_noise:
            panels += [("10 calls, ancestral steps", ancestral), ("10 calls, DDIM", ddim_samples)]
plot_points(panels)

### One draw of the noise at every number of calls

The notes' subsection "One image for every draw of noise" says that with the weights and the levels fixed, a run of deterministic DDIM is a fixed map from the starting noise to a sample, and that runs with different numbers of calls approximate the same continuous path. The cell takes the samples of 1000 calls of each sampler, with the exact prediction, and counts the share of samples that end on the same side of the middle line with 10, 20 and 50 calls. Keep the output: the report reads it.

In [ ]:
reference = {
    "ancestral": sample_ancestral(exact_noise, schedule, x_start, even_levels(1000), torch.Generator().manual_seed(7))[0],
    "DDIM": sample_ddim(exact_noise, schedule, x_start, even_levels(1000))[0],
}
print("calls   ancestral steps   DDIM     share of samples on the same side as with 1000 calls")
for calls in (10, 20, 50):
    fewer = {
        "ancestral": sample_ancestral(exact_noise, schedule, x_start, even_levels(calls), torch.Generator().manual_seed(7))[0],
        "DDIM": sample_ddim(exact_noise, schedule, x_start, even_levels(calls))[0],
    }
    same = {name: ((fewer[name][:, 0] > 0) == (reference[name][:, 0] > 0)).float().mean().item() for name in fewer}
    print(f"{calls:5d}   {same['ancestral']:15.3f}   {same['DDIM']:.3f}")

## 4 · Training by flow matching

The Lecture 5 notes choose the path of each pair before any training. The pair is a training point $\mathbf x_0$ and an independent draw $\boldsymbol\epsilon$ of standard Gaussian noise, and the path is the straight line
$$\mathbf x_t=(1-t)\,\mathbf x_0+t\,\boldsymbol\epsilon,\qquad 0\le t\le1,$$
with the time $t$ running from the data at 0 to the noise at 1. Its velocity $\boldsymbol\epsilon-\mathbf x_0$ is the same at every time. Lines from different pairs cross, so the point and the time do not say which line a sampler is on, and the subsection "The mean velocity at each point" shows that a regression on the lines' velocities learns their mean at each point and time, the mean field $\mathbf u(\mathbf x,t)=\mathbb E[\boldsymbol\epsilon-\mathbf x_0\mid\mathbf x_t=\mathbf x]$. The subsection "The loss" gives the loss, the squared error between the velocity of the line and the network's velocity, and says that it cannot reach zero: its floor is the spread of the velocities of the lines around their mean.

### The floor at the two ends of the line · 4 points

On the two clusters, the clean point $\mathbf x_0$ is $(-2,0)$ or $(2,0)$ with equal chances plus Gaussian noise of standard deviation 0.5 in each coordinate, and $\boldsymbol\epsilon$ is independent standard Gaussian noise. Work coordinate by coordinate, and give each answer as a pair `(first coordinate, second coordinate)` of plain numbers:

1. `zero_loss`: the expected loss of a network that returns the velocity zero everywhere, the average of $(\epsilon-x_0)^2$ in each coordinate. The cross term averages to zero, because the noise is independent of the clean point and has mean zero.
2. `floor_at_one`: the floor at time 1. There the point is the noise itself, $\mathbf x_1=\boldsymbol\epsilon$, so the noise is known exactly and says nothing about the clean point. The floor is the average squared deviation of the velocity from its conditional mean, its conditional variance, not a standard deviation.
3. `floor_near_zero`: the floor, again a variance, as the time approaches 0. There the point is almost the clean point, which the notes' subsection "The loss" says leaves the noise as the unknown part of the velocity.

**Write the three lines in the cell below.**

In [ ]:
# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Compute zero_loss, floor_at_one and floor_near_zero.")
# ================= YOUR CODE ENDS HERE ===================
print("zero velocity:   ", zero_loss, "   mean", sum(zero_loss) / 2)
print("floor at time 1: ", floor_at_one, "   mean", sum(floor_at_one) / 2)
print("floor near 0:    ", floor_near_zero, "   mean", sum(floor_near_zero) / 2)

In [ ]:
from notebook_support import check_line_ends

check_line_ends(zero_loss, floor_at_one, floor_near_zero)

### Task 2 · the loss of flow matching · 7 points

Write `flow_matching_loss(model, x0, generator)`, steps 2 to 5 of the notes' algorithm "train by flow matching". It receives a velocity network called as `model(x_t, t)`, a batch of clean data with the batch on the first axis, of shape $(B,2)$ for the two clusters or $(B,1,28,28)$ for digits, and a random generator, and returns the loss on the batch. Two choices are fixed so that a seeded run gives the same numbers as the course's, and the tests check them:

- Draw the times first, one per row, uniformly between 0 and 1, with `torch.rand`, `generator=generator` and `device=x0.device`. Then draw the noise, of the shape of `x0`, with `torch.randn` and the same generator and device.
- Call the network once, with the times as a float tensor of shape $(B,)$, and return the mean of the squared differences over every number of the batch, a tensor with one value that keeps its gradient, so that the zero velocity scores about 3.25 on the two clusters.

The times have shape $(B,)$; give them one trailing axis of size one for every axis of `x0` after the first when you form the points, as Task 1 of Homework 4 did with the numbers of the schedule. The supplied `train_run` calls your loss when its objective is `"velocity"`.

In [ ]:
import torch


def flow_matching_loss(model, x0, generator):
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 2: implement flow_matching_loss")
    # ================= YOUR CODE ENDS HERE ===================
print("function defined")

In [ ]:
from notebook_support import run_task2_tests

run_task2_tests(flow_matching_loss)

### Two reference losses

Before anything is trained, the cell measures your loss for two velocities that need no training. The first is zero. The second is `ExactVelocity` in `cs687/toy.py`, the mean field of the two clusters, computed exactly from the density of the points on the lines as $(\mathbf x-\mathbb E[\mathbf x_0\mid\mathbf x_t=\mathbf x])/t$, the form the notes derive. It is what a regression on the velocity can at best learn, so its expected loss is the floor.

In [ ]:
from cs687 import ExactVelocity

exact_velocity = ExactVelocity()
reference_points = sample_clusters(50_000, torch.Generator().manual_seed(1))
with torch.no_grad():
    zero_velocity_loss = flow_matching_loss(lambda x, t: torch.zeros_like(x), reference_points, torch.Generator().manual_seed(2)).item()
    velocity_floor = flow_matching_loss(exact_velocity, reference_points, torch.Generator().manual_seed(2)).item()
print(f"the zero velocity:             {zero_velocity_loss:.4f}")
print(f"the exact mean field, floor:   {velocity_floor:.4f}")

### The toy run

The notes' subsection "The trainer beside the earlier one" trains the network of Lecture 3 with the new target. `ToyVelocity` in `cs687/networks.py` is the small network of Homework 4 with one change: the time between 0 and 1 enters as the sinusoidal vector of $1000\,t$, which gives it the range of frequencies it had for the levels. The supplied `train_run` trains it with your loss for 4,000 steps of Adam at the learning rate 0.001 on batches of 512, the settings of Homework 4, in a few seconds. Single batches fall below the floor by chance; the floor bounds the expected loss, which the running mean follows. Keep the output: the report reads it.

In [ ]:
from cs687 import ToyVelocity
from notebook_support import plot_loss_against_floor

torch.manual_seed(0)
velocity_network = ToyVelocity()
started = time.time()
velocity_curve = train_run(velocity_network, sample_clusters, "velocity", steps=4000, batch_size=512, learning_rate=1e-3,
                           generator=torch.Generator().manual_seed(13), print_every=1000)
print(f"trained in {time.time() - started:.1f} s; mean loss of the last 500 steps {sum(velocity_curve[-500:]) / 500:.4f}, "
      f"floor {velocity_floor:.4f}")
plot_loss_against_floor(velocity_curve, velocity_floor, zero_velocity_loss)

### Where the floor comes from

Each row of a batch draws its own time, so the loss mixes all times. The first table measures the loss of your velocity network and of the exact mean field separately in ten bands of time, on 20,000 fresh points per band. The second repeats Homework 4's table for the noise loss, in ten bands of levels, with the noise predictor of section 2 and the exact noise prediction. In both tables the first row is nearest the data and the last nearest the noise. Keep the output: the report reads both tables.

In [ ]:
from cs687 import noisy_at_level

band_generator = torch.Generator().manual_seed(14)
print("flow matching, per number")
print("times          your network   exact mean field")
with torch.no_grad():
    for band in range(10):
        x0 = sample_clusters(20_000, band_generator)
        t = (band + torch.rand(20_000, generator=band_generator)) / 10
        eps = torch.randn(x0.shape, generator=band_generator)
        x_t = (1 - t[:, None]) * x0 + t[:, None] * eps
        yours = ((velocity_network(x_t, t) - (eps - x0)) ** 2).mean().item()
        best = ((exact_velocity(x_t, t) - (eps - x0)) ** 2).mean().item()
        print(f"{band / 10:.1f} to {(band + 1) / 10:.1f}     {yours:12.4f}   {best:16.4f}")

print()
print("the noise loss of Homework 4, per number")
print("levels         your network   exact prediction")
with torch.no_grad():
    for low in range(1, 1001, 100):
        x0 = sample_clusters(20_000, band_generator)
        t = torch.randint(low, low + 100, (20_000,), generator=band_generator)
        eps = torch.randn(x0.shape, generator=band_generator)
        x_t = noisy_at_level(x0, t, eps, schedule)
        yours = ((noise_network(x_t, t) - eps) ** 2).mean().item()
        best = ((exact_noise(x_t, t) - eps) ** 2).mean().item()
        print(f"{low:4d} to {low + 99:<4d}   {yours:12.4f}   {best:16.4f}")

## 5 · Sampling with Euler steps

### Task 3 · one Euler step · 3 points

The notes' section "Sampling with Euler steps" generates a sample by following the network's velocity from a draw of the noise at time 1 down to time 0. An Euler step keeps the velocity at the start of the step for the whole step, and time runs down during sampling.

Write `euler_step(model, x_t, t, s)`. It receives a velocity network called as `model(x_t, times)`, a batch `x_t` whose rows are all at the time `t`, and the current time `t` and the next time `s` as Python floats with $0\le s<t\le1$, and it returns the batch at time `s`, of the shape of `x_t`. Call the network once, with `times` a float tensor of $B$ copies of `t` on the device of `x_t`; `torch.full` makes it. The supplied loop `sample_euler` in `cs687/samplers.py` calls your step down a list of times under `torch.no_grad()` and counts the calls, and `even_times(calls)` gives the evenly spaced times from 1 down to 0.

In [ ]:
import torch


def euler_step(model, x_t, t, s):
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 3: implement euler_step")
    # ================= YOUR CODE ENDS HERE ===================
print("function defined")

In [ ]:
from notebook_support import run_task3_tests

run_task3_tests(euler_step)

### The exact field and your network, from 1 to 50 calls

The cell samples from the same 2,000 draws with Euler steps at evenly spaced times, with 1, 2, 5, 10 and 50 calls, first with the exact mean field and then with your velocity network, and draws both rows. With one call every sample of the exact field lies on the middle line, where none counts as on the right.

In [ ]:
from cs687 import even_times, sample_euler

for source, model in (("the exact mean field", exact_velocity), ("your velocity network", velocity_network)):
    print(f"--- with {source}")
    panels = []
    for calls in (1, 2, 5, 10, 50):
        samples, used = sample_euler(model, x_start, even_times(calls))
        show("Euler steps", cluster_statistics(samples), used)
        panels.append((f"{calls} call{'s' if calls > 1 else ''}, {'exact' if model is exact_velocity else 'your network'}", samples))
    plot_points(panels)

## 6 · One sampler seen twice

The notes' subsection "The update of the previous lecture on a straight line" carries out the update of DDIM on the line. The point on a line can be written from either end,
$$\mathbf x_t=\mathbf x_0+t\,(\boldsymbol\epsilon-\mathbf x_0)=\boldsymbol\epsilon-(1-t)\,(\boldsymbol\epsilon-\mathbf x_0),$$
and putting the network's velocity $\mathbf v_{\boldsymbol\theta}$ in the place of the velocity of the line gives an estimate $\hat{\mathbf x}_0$ of the clean point and an estimate $\hat{\boldsymbol\epsilon}$ of the noise. The update of DDIM then forms the point at the lower time $s$ by mixing the two estimates with the weights of the line at that time, $\mathbf x_s=(1-s)\,\hat{\mathbf x}_0+s\,\hat{\boldsymbol\epsilon}$.

### DDIM on the line by hand · 4 points

Take one coordinate at the time $t=0.75$ with the value $x_t=0.8$, and suppose the network returns the velocity 1.2 there. The next time is $s=0.25$. Compute:

1. `x0_hat_line` and `eps_hat_line`: the estimates of the clean value and of the noise.
2. `x_s_mixed`: the value at time 0.25 from mixing the two estimates with the weights of the line.
3. `x_s_euler`: the value at time 0.25 from one Euler step.

**Write the four lines in the cell below.**

In [ ]:
# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Compute x0_hat_line, eps_hat_line, x_s_mixed and x_s_euler.")
# ================= YOUR CODE ENDS HERE ===================
print(f"x0_hat_line {x0_hat_line:.4f}   eps_hat_line {eps_hat_line:.4f}")
print(f"x_s_mixed {x_s_mixed:.4f}   x_s_euler {x_s_euler:.4f}")

In [ ]:
from notebook_support import check_line_step

check_line_step(x0_hat_line, eps_hat_line, x_s_mixed, x_s_euler)

### Task 4 · the estimates from a velocity · 3 points

Write `estimates_from_velocity(v, x_t, t)`. It receives a batch of velocities, the batch of points they were computed at, and a float tensor `t` of their $B$ times, one per row, and returns the tuple `(x0_hat, eps_hat)`, in that order, each of the shape of `x_t`. Leave the tensor `x_t` unchanged. Give the times one trailing axis of size one for every axis of `x_t` after the first.

In [ ]:
import torch


def estimates_from_velocity(v, x_t, t):
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 4: implement estimates_from_velocity")
    # ================= YOUR CODE ENDS HERE ===================
print("function defined")

In [ ]:
from notebook_support import run_task4_tests

run_task4_tests(estimates_from_velocity)

### The two stages against the Euler step

The cell takes 2,000 points at the time 0.75, asks your velocity network for its velocity there, and moves the points to the time 0.25 twice: with the two stages of DDIM on the line, built from your Task 4, and with one Euler step, your Task 3. It prints the largest difference between the two.

In [ ]:
with torch.no_grad():
    points = 0.25 * sample_clusters(2000, torch.Generator().manual_seed(15)) + 0.75 * torch.randn(2000, 2, generator=torch.Generator().manual_seed(16))
    times = torch.full((2000,), 0.75)
    x0_hat, eps_hat = estimates_from_velocity(velocity_network(points, times), points, times)
    two_stages = (1 - 0.25) * x0_hat + 0.25 * eps_hat
    one_euler_step = euler_step(velocity_network, points, 0.75, 0.25)
print(f"largest difference between the two stages of DDIM and the Euler step: {(two_stages - one_euler_step).abs().max().item():.1e}")

### Every noise schedule as a path

The notes' subsection "Every noise schedule as a path" writes both paths as $\mathbf x_t=a_t\,\mathbf x_0+b_t\,\boldsymbol\epsilon$. Divided by the weight $a_t$ of the clean image, each becomes the rescaled image $\mathbf x_0+\sigma\,\boldsymbol\epsilon$ with the noise scale $\sigma=b_t/a_t$, and two paths carry the same amount of noise where their noise scales are equal. For the noising process, $\sigma$ at a level is the noise scale $\sigma_t$ of Lecture 4. With the exact field, every such path gives the same map from noise to samples, and DDIM on any of them is the Euler step of one ODE; what a choice of path changes is where evenly spaced steps fall.

### Where four calls fall · 4 points

With four calls, DDIM on evenly spaced levels calls the network at the levels 1000, 750, 500 and 250, and the Euler sampler on evenly spaced times calls it at the times 1, 0.75, 0.5 and 0.25. Compute, as lists of four plain numbers, with the numbers of `exact_schedule`:

1. `ddim_sigmas`: the noise scales of the four levels.
2. `ddim_times`: the times of the straight line with the same noise scales.
3. `line_sigmas`: the noise scales of the straight line at its own four times. Write `math.inf` for the noise scale at time 1, where the weight of the clean point is zero.

The last entry of each list is the last call of that sampler.

**Write the three lines in the cell below.**

In [ ]:
# ================= YOUR CODE STARTS HERE =================
raise NotImplementedError("Compute ddim_sigmas, ddim_times and line_sigmas.")
# ================= YOUR CODE ENDS HERE ===================
print("DDIM:  levels  1000, 750, 500, 250   noise scales", [round(v, 3) for v in ddim_sigmas], "  times", [round(v, 3) for v in ddim_times])
print("line:  times   1, 0.75, 0.5, 0.25    noise scales", [round(v, 3) for v in line_sigmas])

In [ ]:
from notebook_support import check_four_calls

check_four_calls(schedule, ddim_sigmas, ddim_times, line_sigmas)

### Task 5 · the time of a level on the line · 4 points

Write `line_time(levels, schedule)`. It receives a long tensor of levels from 0 to $T$, of any shape, and the schedule, and returns a float tensor of the shape of `levels` holding, for each level, the time of the straight line whose noise scale equals the noise scale of the noising process at that level. Level 0 has the noise scale 0 and the time 0.

In [ ]:
import torch


def line_time(levels, schedule):
    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError("Task 5: implement line_time")
    # ================= YOUR CODE ENDS HERE ===================
print("function defined")

In [ ]:
from notebook_support import run_task5_tests

run_task5_tests(line_time)

### DDIM as Euler steps at other times

The cell runs DDIM on evenly spaced levels with the exact noise prediction, and the Euler sampler with the exact mean field at the times that your Task 5 gives for the same levels, from the same draws, and prints the largest difference between the two sets of samples in any coordinate. Then it repeats the Euler run at the times the level divided by 1000 would give. Last, it runs your velocity network at DDIM's converted times and at its own evenly spaced times with two calls. Keep the output: the report reads it.

In [ ]:
print("calls   largest difference from DDIM:  at the times of Task 5   at the level divided by 1000")
for calls in (2, 3, 5, 10):
    levels = even_levels(calls)
    ddim_samples, _ = sample_ddim(exact_noise, schedule, x_start, levels)
    converted, _ = sample_euler(exact_velocity, x_start, line_time(torch.tensor(levels), schedule).tolist())
    divided, _ = sample_euler(exact_velocity, x_start, [level / 1000 for level in levels])
    print(f"{calls:5d}   {(converted - ddim_samples).abs().max().item():47.4f}   {(divided - ddim_samples).abs().max().item():28.4f}")

print()
two_call_times = line_time(torch.tensor(even_levels(2)), schedule).tolist()
print("DDIM's two calls on the time of the line:", [round(t, 3) for t in two_call_times[:-1]], "  the line's own:", even_times(2)[:-1])
show("your velocity network, DDIM's times", cluster_statistics(sample_euler(velocity_network, x_start, two_call_times)[0]), 2)
show("your velocity network, its own times", cluster_statistics(sample_euler(velocity_network, x_start, even_times(2))[0]), 2)
show("DDIM, the exact prediction", cluster_statistics(sample_ddim(exact_noise, schedule, x_start, even_levels(2))[0]), 2)

### Where the choice of cluster is made

The notes' subsection "What a sampler needs from a path" says that with few calls the sample is the estimate of the clean point made at the last call. The cell follows the paths of the 2,000 draws with 1,000 Euler steps, with the exact mean field and with your velocity network, and at ten times prints the mean distance from the middle line of the estimate of the clean point, which your Task 4 forms from the velocity. The data's mean distance is about 2. It also prints the times of the calls of each sampler with two, three and five calls, on the time of the line. Keep the output: the report reads it.

In [ ]:
MARKED = (0.95, 0.86, 0.77, 0.67, 0.59, 0.5, 0.42, 0.33, 0.2, 0.1)
fine = even_times(1000)
along = {}
with torch.no_grad():
    for source, model in (("exact", exact_velocity), ("network", velocity_network)):
        x = x_start
        for t, s in zip(fine[:-1], fine[1:]):
            times = torch.full((len(x),), t)
            v = model(x, times)
            if round(t, 3) in MARKED:
                along[(source, round(t, 3))] = estimates_from_velocity(v, x, times)[0][:, 0].abs().mean().item()
            x = x - (t - s) * v
print("time   estimate of the clean point, mean distance from the middle:  exact field   your network")
for t in MARKED:
    print(f"{t:4.2f}   {along[('exact', t)]:63.3f}   {along[('network', t)]:12.3f}")
print()
print("the times of the calls on the time of the line, the last call last")
for calls in (2, 3, 5):
    ddim_calls = line_time(torch.tensor(even_levels(calls)), schedule)[:-1].tolist()
    line_calls = even_times(calls)[:-1]
    print(f"{calls} calls   DDIM {[round(t, 3) for t in ddim_calls]}   the line {[round(t, 3) for t in line_calls]}")

## 7 · Quality against calls

This is the measurement of Project 2. The cell runs the three samplers from the same 2,000 draws at 1, 2, 3, 5, 10, 20, 50, 100 and 1000 calls: the larger ancestral steps and DDIM on evenly spaced levels, and the Euler sampler on evenly spaced times. It runs them once with the exact fields, the exact noise prediction for the first two and the exact mean field for the third, so that the only error is the error of the steps, and once with your two networks. It prints the three statistics and draws the mean distance and the spread against the number of calls. It takes under a minute. Keep the output: the report reads it.

In [ ]:
from notebook_support import plot_quality_against_calls

CALLS = [1, 2, 3, 5, 10, 20, 50, 100, 1000]
MODELS = {
    "exact": {"ancestral": exact_noise, "DDIM": exact_noise, "Euler": exact_velocity},
    "network": {"ancestral": noise_network, "DDIM": noise_network, "Euler": velocity_network},
}

def run_sampler(sampler, model, calls):
    if sampler == "ancestral":
        return sample_ancestral(model, schedule, x_start, even_levels(calls), torch.Generator().manual_seed(7))
    if sampler == "DDIM":
        return sample_ddim(model, schedule, x_start, even_levels(calls))
    return sample_euler(model, x_start, even_times(calls))

started = time.time()
quality = {}
for source, models in MODELS.items():
    for sampler, model in models.items():
        rows = []
        for calls in CALLS:
            samples, used = run_sampler(sampler, model, calls)
            assert used == calls
            rows.append(cluster_statistics(samples))
        quality[(sampler, source)] = rows

for source, title in (("exact", "the exact fields"), ("network", "your networks")):
    print(f"--- {title}: share right, mean distance, spread within")
    print("calls   " + "".join(f"{sampler:>26s}" for sampler in ("ancestral", "DDIM", "Euler")))
    for i, calls in enumerate(CALLS):
        cells = "".join(f"    {quality[(sampler, source)][i]['share_right']:.3f} {quality[(sampler, source)][i]['mean_distance']:6.3f} "
                        f"{quality[(sampler, source)][i]['spread_within']:6.3f}" for sampler in ("ancestral", "DDIM", "Euler"))
        print(f"{calls:5d}{cells}")
print(f"the data:   share right {data_stats['share_right']:.3f}, mean distance {data_stats['mean_distance']:.3f}, "
      f"spread within {data_stats['spread_within']:.3f}        measured in {time.time() - started:.0f} s")
plot_quality_against_calls(CALLS, quality, data_stats)

### Your noise predictor at the first calls

With the exact fields, DDIM and the Euler sampler agree from five calls on. With your networks, DDIM and the larger ancestral steps, the two samplers of the noise predictor, go wrong with few calls, while the velocity network stays close to its exact field: with one call they throw the samples out to a mean distance of about 7.5, off the top of the plot, and with two and three calls the share on the right is far from one half, so most samples land in one cluster. The reason is where the samplers form the estimate of the clean point; both samplers of the noise predictor form it as DDIM does. The Lecture 4 notes' identity $\bar{\mathbf x}_t=\hat{\mathbf x}_0+\sigma_t\hat{\boldsymbol\epsilon}$, at a level $t$, holds for any predicted noise, so DDIM's estimate is $\hat{\mathbf x}_0=\bar{\mathbf x}_t-\sigma_t\hat{\boldsymbol\epsilon}$, and an error in the predicted noise reaches it multiplied by the noise scale, about 157 at level 1000. On the line the estimate at the time $t$ is $\mathbf x_t-t\,\mathbf v_{\boldsymbol\theta}$, so an error in the velocity reaches it multiplied by the time, at most 1. With few calls, each step travels most of the way towards the estimate of the moment, so a wrong estimate at the first calls moves the samples a long way. The cell measures both errors against the exact fields on noisy points of the two clusters. This section is not assessed; it explains the difference between the two halves of the measurement.

In [ ]:
error_generator = torch.Generator().manual_seed(17)
print("level   error of the predicted noise   noise scale   error of the estimate of the clean point")
with torch.no_grad():
    for level in (1000, 800, 600, 400):
        x0 = sample_clusters(4000, error_generator)
        levels = torch.full((4000,), level)
        x_t = noisy_at_level(x0, levels, torch.randn(x0.shape, generator=error_generator), schedule)
        error = (noise_network(x_t, levels) - exact_noise(x_t, levels)).pow(2).mean().sqrt().item()
        sigma = math.sqrt((1 - schedule.alpha_bar[level].item()) / schedule.alpha_bar[level].item())
        print(f"{level:5d}   {error:28.3f}   {sigma:11.1f}   {error * sigma:40.3f}")
print()
print("time    error of the velocity   error of the estimate of the clean point")
with torch.no_grad():
    for t in (1.0, 0.9, 0.77, 0.5):
        x0 = sample_clusters(4000, error_generator)
        times = torch.full((4000,), t)
        x_t = (1 - t) * x0 + t * torch.randn(x0.shape, generator=error_generator)
        error = (velocity_network(x_t, times) - exact_velocity(x_t, times)).pow(2).mean().sqrt().item()
        print(f"{t:4.2f}   {error:21.3f}   {error * t:40.3f}")

## 8 · Digits

The same loss trains the network of the digits. `VelocityUNet` in `cs687/networks.py` is the small U-Net of Homework 4 with the time entering as the vector of $1000\,t$. The run trains it with your loss on the 60,000 training digits, in batches of 128 with Adam at the learning rate 0.0002, for 10,000 steps, the settings of Homework 4's digit run.

The run needs a GPU runtime. The training takes about 17 minutes on a Colab T4 and about 7 minutes on an L4. When it finishes, the trained parameters are saved in `runs/digits_velocity.pt`, and running the cell again restores them in seconds instead of training again, as long as the runtime's disk is still there.

In [ ]:
from cs687 import VelocityUNet, load_training_digits

digits_on_device = load_training_digits().to(device)

def draw_digits(n, generator):
    rows = torch.randint(0, len(digits_on_device), (n,), generator=generator, device=generator.device)
    return digits_on_device[rows]

torch.manual_seed(0)
velocity_unet = VelocityUNet().to(device)
print(f"VelocityUNet: {sum(p.numel() for p in velocity_unet.parameters()):,} parameters, on {device}")
started = time.time()
digit_curve = train_run(
    velocity_unet, draw_digits, "velocity", steps=10_000, batch_size=128, learning_rate=2e-4,
    generator=torch.Generator(device=device).manual_seed(7), checkpoint_path="runs/digits_velocity.pt",
    print_every=1000, shortened_by_grader=True,
)
print(f"the training cell took {time.time() - started:.0f} s")

### Two samplers on digits

The cell starts from eight draws of the noise and samples them with the Euler sampler on your velocity network, and with DDIM on the course's noise predictor of the digits, the network of Homework 4's digit run, which `load_course_digit_network` reads from `weights/digit_noise_predictor.pt`. Each sampler runs with 1, 2, 5, 10 and 50 calls, every row from the same eight draws. The two networks were trained separately, so the two blocks differ in their training as well as in where their steps fall.

In [ ]:
from cs687 import load_course_digit_network
from notebook_support import plot_digit_rows

noise_unet = load_course_digit_network(device)
digit_schedule = schedule.to(device)
digit_start = torch.randn(8, 1, 28, 28, generator=torch.Generator(device=device).manual_seed(8), device=device)
started = time.time()
euler_rows, ddim_rows = [], []
for calls in (1, 2, 5, 10, 50):
    euler_rows.append((f"{calls} call{'s' if calls > 1 else ''}", sample_euler(velocity_unet, digit_start, even_times(calls))[0]))
    ddim_rows.append((f"{calls} call{'s' if calls > 1 else ''}", sample_ddim(noise_unet, digit_schedule, digit_start, even_levels(calls))[0]))
print(f"sampled in {time.time() - started:.0f} s on {device}")
plot_digit_rows(euler_rows, "Euler steps on your velocity network")
plot_digit_rows(ddim_rows, "DDIM on the course's noise predictor")

## 9 · Run the public tests

The immediate checks above give feedback after each guided calculation and programming task. This final cell repeats all four calculation checks and runs all 25 public tests against the five task implementations currently loaded in the notebook. Always run it before submitting your notebook. It does not require you to copy anything into the repository's `.py` files. Staff grading uses a separate, staff-controlled test suite.

In [ ]:
from notebook_support import (
    check_ddim_by_hand,
    check_four_calls,
    check_line_ends,
    check_line_step,
    run_all_public_tests,
)

check_ddim_by_hand(schedule, x0_hat_600, x_300, sigma_600, sigma_300, x_300_rescaled)
check_line_ends(zero_loss, floor_at_one, floor_near_zero)
check_line_step(x0_hat_line, eps_hat_line, x_s_mixed, x_s_euler)
check_four_calls(schedule, ddim_sigmas, ddim_times, line_sigmas)

run_all_public_tests(ddim_step, flow_matching_loss, euler_step, estimates_from_velocity, line_time)

## 10 · Report

Use the four cells below to explain what the notebook shows. Do not paste whole output tables or repeat the code line by line. Support your explanations with what your own runs show. These report cells are part of the notebook submission, and so are the outputs of the demonstrations they rest on.

### 1 · DDIM against the larger ancestral steps · 15 points

In section 3 you sampled the two clusters from the same 2,000 draws of the noise with the larger ancestral steps and with DDIM, your Task 1, at 1000, 50, 20, 10 and 5 calls, and you counted how many samples end on the same side of the middle with 10, 20 and 50 calls as with 1000. The Lecture 4 notes' part "The cost of a thousand steps" and its subsections "Large steps from the estimate of the clean image", "One image for every draw of noise" and "The error of few steps" explain what you saw. Answer from the runs with the exact prediction.

- With 50, 20 and 10 calls, how did the samples of the larger ancestral steps differ from the data and from DDIM's, in their spread within a half and their mean distance from the middle? Then explain why a step across many levels pulls the samples towards the middle, in terms of the distribution of the image at the lower level given the current image and the single Gaussian that the sampler draws from.
- What does a DDIM step hold fixed between the current level and the lower level? For what data is one DDIM step from level 1000 to level 0 exact, and why?
- Why did every DDIM sample end on the same side with 10 calls as with 1000, while about half of the ancestral samples did not? Answer in terms of when each sampler draws noise.

Write approximately 150–250 words.

**Response:**

_Replace this line with your response._

### 2 · The floor of flow matching · 15 points

In section 4 you trained the small network by flow matching with your Task 2 and plotted its loss against the loss of the exact mean field, the floor. Then you measured both in ten bands of time, beside the noise loss of Homework 4's network in ten bands of levels. In calculation 2 you computed the floor at the two ends of the line. The Lecture 5 notes' subsections "The mean velocity at each point" and "The loss" explain the floor.

- The exact mean field is the conditional mean of the velocity $\boldsymbol\epsilon-\mathbf x_0$ given the point and the time. Why can no network reach a lower expected loss than it? Say what varies among the targets that share one input.
- How does the floor in the bands of time below 0.4 compare with the bands above 0.6? Then explain the two ends of the line with calculation 2: which part of the velocity the point leaves unknown as the time approaches 0 and which at time 1, and why. The shape of the floor between the two ends is not asked.
- The noise loss of Homework 4 had its floor near zero in the bands nearest the noise. Why is the floor of the noise loss small where the floor of flow matching is largest?

Write approximately 150–250 words.

**Response:**

_Replace this line with your response._

### 3 · One sampler seen twice · 15 points

In section 6 you carried out one step of DDIM on the straight line by hand in calculation 3 and on your network with Task 4, and saw it give the Euler step of Task 3. Then you converted DDIM's levels into times of the line with Task 5, and with the exact fields the Euler sampler at those times reproduced DDIM's samples, while the Euler sampler at the level divided by 1000 did not. The Lecture 5 notes' subsections "The update of the previous lecture on a straight line" and "Every noise schedule as a path" explain this.

- Write the two estimates that DDIM's update forms from the velocity on the line. Then explain why the update is exactly the Euler step, in terms of what it holds fixed during the step and what the velocity of a line is.
- Why do the times of Task 5 reproduce DDIM while the level divided by 1000 does not? Answer in terms of the noise scale, with one level as the example: its noise scale, its time from Task 5, the level divided by 1000, and the noise scale of the line at that time, from calculation 4 or from section 6.
- The two samplers follow one ODE, yet in section 7 they give different samples with two calls. What differs between the two runs, if not the field, and why does that difference change the samples?

Write approximately 150–250 words.

**Response:**

_Replace this line with your response._

### 4 · Quality against calls · 15 points

In section 7 you measured the three samplers from 1000 calls down to one, with the exact fields and with your networks, and in section 6 you followed the estimate of the clean point along paths of very small steps and printed the time of each sampler's last call. The Lecture 5 notes' subsection "The mean velocity at each point", the section "Sampling with Euler steps" and the subsections "A field that bends" and "What a sampler needs from a path" explain the curves. Answer from the runs with the exact fields.

- With one call, every sample of the Euler sampler landed on the middle line. Explain why, from the mean field at time 1.
- With few calls, what is a sample of the Euler sampler, in terms of its last call, and why? With two calls, the samples of the Euler sampler lie further from the middle than those of DDIM. Explain why, from the time of each sampler's last call and your table of the estimate of the clean point along the paths.
- From five calls on, the two samplers agree. Why does the placing of the steps matter less there, from where the five calls of each fall? The paths still bend although every line of training is straight. Explain where that bend comes from: how the estimate of the clean point moves as the time falls from 1 towards 0, and why.

Write approximately 200–300 words.

**Response:**

_Replace this line with your response._

## 11 · Finishing the homework

Submit only this notebook. Before uploading it to Moodle, rename it to `homework05_colab_STUDENTNUMBER.ipynb`, replacing `STUDENTNUMBER` with your student number. For example: `homework05_colab_22206543.ipynb`.

Before submitting, restart the Colab runtime and run the notebook from top to bottom. The digit run of section 8 is restored from its checkpoint in seconds if the session's disk is still there, and takes the time stated in section 8 on a GPU runtime otherwise. If a run trained again, check that every value and phrase your report quotes matches the new outputs. Confirm that the final check reports that all four guided calculations and all 25 public tests passed, and make sure none of the four report cells still contains its placeholder. Save the notebook in Colab with its outputs, rename it, download it with File > Download > Download .ipynb, and upload that exact file, and only that file, to the Moodle assignment for Homework 5.

The outputs of sections 2 to 8, the printed tables, the plots and the digits, are the evidence your report is read against, so keep them in the saved notebook. Grading runs your notebook again from a clean environment to check the calculations and the five tasks, with a short digit run in place of yours. The report is read against the outputs you saved.